# Analysis

## Setting parameters

In [63]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [64]:
from quickrules.data_handling import calculate_score, count_all_rules, count_all_attributes, bold
from pathlib import Path
from quickrules.data_handling import balanced_accuracy_score
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd
import numpy as np

In [65]:
data_sets = [  # actual set for QR
    'australian',
     # 'automobile', # cat feats
     'bands',
     'bupa',
     'cleveland',
     # 'contraceptive', # no rules found
     # 'crx', # cat feats
     'dermatology',
     'ecoli',
     # 'german', # cat feats
     'glass',   
     # 'haberman', # no rules found
     'heart',
     'ionosphere',
     # 'mammographic',  # no rules found
     'pima',
     # 'saheart', # cat feats
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     'vowel',
     'wine',
     'winequality-red',
     'wisconsin',
     'yeast'
]

In [66]:
len(data_sets)

18

In [67]:
data_base = Path.cwd() / 'keel-data'
include_sets = data_sets
metric = balanced_accuracy_score  # balanced_
scores = {}
rules = {}
attributes_mean = {}
attributes_std = {}
results_base = Path.cwd() / 'results'

## Loading WEKA results
MODLEM, FURIA, RIPPER

In [68]:
weka_folder = Path.cwd() / 'weka-results'
weka_models = {
    'modlem':'&',
    'furia':'and',
    'ripper':'and'
}
acc_type = 'balanced-accuracy'  # 'weka-accuracy.csv'

In [69]:
for name, connection in weka_models.items():
    model_folder = weka_folder / f"{name}-files"
    file = model_folder / f"{name}-{acc_type}.csv"
    scores[name] = pd.read_csv(weka_folder / file, header=None, index_col=0).to_dict()[1]
    rules[name] = {}
    attributes_mean[name] = {}
    attributes_std[name] = {}
    for file in model_folder.iterdir():
        if file.name[-3:] == 'txt':
            short_name = file.name[:-4]
            with open(file, 'r') as f:
                line = f.readline()
                nrs = []
                while len(line) > 4:
                    nrs.append(line.count(connection) + 1)
                    line = f.readline()
            rules[name][short_name] = len(nrs)
            attributes_mean[name][short_name] = np.average(nrs)
            attributes_std[name][short_name] = np.std(nrs)

## Loading QuickRules results

In [70]:
qr_models = {
    'qr': results_base / 'no-prune-results' / 'close-min-max-combo-results',
}

In [71]:
for model, path in qr_models.items():
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=include_sets,
        metric=metric
    )
    rules[model] = count_all_rules(
        Path.cwd() / results_base / path,
        include=include_sets
    )
    attributes_mean[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=data_sets
    )
    attributes_std[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=data_sets,
        metric=np.std
    )

## FRRI Models

In [72]:
frri_models = {
    'frri' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
}

std = {}

In [73]:
for model, path in frri_models.items():
    scores[model] = calculate_score(
        data_folder=Path.cwd() / 'keel-data',
        results_folder=path,
        metric=metric,
        include=data_sets,
        label_encoding=True
    )
    # std[model] = calculate_score(
    #     data_folder=Path.cwd() / 'keel-data',
    #     results_folder=path,
    #     metric=metric,
    #     include=data_sets,
    #     label_encoding=True,
    #     aggregation_function=np.std
    # )
    rules[model] = count_all_rules(
        path,
        include=data_sets
    )
    attributes_mean[model] = count_all_attributes(
        path,
        include=data_sets,
        counter='attribute'
    )
    attributes_std[model] = count_all_attributes(
        path,
        include=data_sets,
        counter='attribute',
        metric=np.std
    )

## Tables

In [98]:
names = list(scores.keys())
names = ['frri', 'modlem', 'qr', 'furia', 'ripper']

In [99]:
main_folder = 'paper_tables_furripper_std' # 'balanced_acc_incl'  # 'normal_acc'
tables_path_base = Path.cwd() / 'tables' / main_folder

In [100]:
table_acc = pd.DataFrame(index=data_sets, columns=names)

In [101]:
for model in names:
    for data_set in data_sets:
    # for data_set, score in scores[model].items():
        table_acc.loc[data_set, model] = scores[model][data_set] # f"{scores[model][data_set]} +- {std[model][data_set] if model in std.keys() else 0}"

In [102]:
table_acc.loc['mean'] = table_acc.mean(axis='rows', skipna=True)

In [103]:
table_acc

,frri,modlem,qr,furia,ripper
australian,0.787517,0.861,0.865348,0.8645,0.852
bands,0.675141,0.6035,0.533045,0.6295,0.5815
bupa,0.600724,0.6525,0.5,0.6575,0.6535
cleveland,0.290397,0.2764,0.208347,0.2452,0.2384
dermatology,0.895363,0.941333,0.521995,0.95,0.94183
ecoli,0.720659,0.512,0.17585,0.529875,0.54475
glass,0.679067,0.518333,0.31381,0.521857,0.494857
heart,0.7675,0.7605,0.785833,0.818,0.7625
ionosphere,0.912468,0.8905,0.658974,0.887,0.892
pima,0.687633,0.6985,0.502775,0.7025,0.6985


In [80]:
bolded_first = table_acc.apply(lambda data: bold(data), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab_acc.txt', escape=False)

NameError: name 'nr' is not defined

In [104]:
table_rule = pd.DataFrame(index=data_sets, columns=names)
for model in names:
    # for data_set, rule_count in rules[model].items():
    for data_set in data_sets:
        table_rule.loc[data_set, model] = rules[model][data_set]  #  rule_count
table_rule.loc['mean'] = table_rule.mean(axis='rows', skipna=True)

In [105]:
table_rule

,frri,modlem,qr,furia,ripper
australian,92.3,121,732.2,6,6
bands,59.0,113,222.7,19,4
bupa,76.7,103,332.4,12,5
cleveland,102.2,95,331.1,7,4
dermatology,20.2,27,90.5,11,8
ecoli,56.8,56,315.0,15,9
glass,51.1,50,225.7,13,8
heart,44.7,62,303.7,15,4
ionosphere,19.3,30,460.5,14,7
pima,125.0,191,824.6,13,4


In [228]:
bolded_first = table_rule.apply(lambda data: bold(data, optimum='min', format_string="%.0f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_10899/3026783303.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)


In [128]:
table_attribute = pd.DataFrame(index=data_sets, columns=names)
for data_set in data_sets:
    means = {}
    for model in names:
        means[model] = attributes_mean[model][data_set]
    for model in names:
        mean = f"\\textbf{{{means[model]:.2f}}}" if means[model] == min(means.values()) else f"{means[model]:.2f}"
        table_attribute.loc[data_set, model] = \
            f"{mean} $\\pm$ {attributes_std[model][data_set]:.1f}"
for model in names:
    table_attribute.loc['mean', model] = \
        f"{np.mean([attributes_mean[model][data_set] for data_set in data_sets]):.2f}"


In [129]:
table_attribute

,frri,modlem,qr,furia,ripper
australian,5.15 $\pm$ 1.5,2.36 $\pm$ 0.9,8.05 $\pm$ 1.9,\textbf{2.17} $\pm$ 0.7,2.33 $\pm$ 0.7
bands,6.29 $\pm$ 2.0,2.11 $\pm$ 0.6,\textbf{1.75} $\pm$ 0.6,5.58 $\pm$ 1.4,4.75 $\pm$ 1.1
bupa,4.20 $\pm$ 1.0,\textbf{2.19} $\pm$ 0.6,4.40 $\pm$ 1.0,3.17 $\pm$ 0.8,2.40 $\pm$ 0.8
cleveland,5.45 $\pm$ 1.4,2.59 $\pm$ 1.0,8.08 $\pm$ 2.0,3.86 $\pm$ 0.8,\textbf{2.50} $\pm$ 0.9
dermatology,4.70 $\pm$ 2.1,2.56 $\pm$ 1.0,6.73 $\pm$ 3.5,2.73 $\pm$ 1.1,\textbf{1.75} $\pm$ 0.8
ecoli,3.85 $\pm$ 0.9,2.34 $\pm$ 0.8,3.82 $\pm$ 1.0,3.00 $\pm$ 0.8,\textbf{2.00} $\pm$ 0.8
glass,3.88 $\pm$ 1.1,\textbf{2.10} $\pm$ 0.8,5.34 $\pm$ 1.5,3.38 $\pm$ 1.1,2.38 $\pm$ 0.9
heart,4.90 $\pm$ 1.3,2.21 $\pm$ 0.7,7.58 $\pm$ 1.9,3.33 $\pm$ 1.0,\textbf{1.50} $\pm$ 0.5
ionosphere,4.53 $\pm$ 2.2,1.53 $\pm$ 0.6,11.65 $\pm$ 5.4,2.71 $\pm$ 1.4,\textbf{1.43} $\pm$ 0.5
pima,5.08 $\pm$ 1.2,\textbf{2.07} $\pm$ 0.7,5.69 $\pm$ 1.3,2.69 $\pm$ 1.0,2.25 $\pm$ 0.8


In [229]:
bolded_first = table_attribute.apply(lambda data: bold(data, optimum='min', format_string="%.2f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_10899/3212901870.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)


## Statistical testing

In [28]:
from scipy import stats
import scikit_posthocs as sph

In [37]:
subject = table_acc

In [38]:
no_mean =  subject.drop(labels = 'mean', axis = 'index')
# no_mean.drop('qr', axis, inplace=True)

In [39]:
no_mean

,lower-new-check,modlem,qr,furia,ripper
australian,0.787517,0.861,0.865348,0.8645,0.852
bands,0.675141,0.6035,0.533045,0.6295,0.5815
bupa,0.600724,0.6525,0.5,0.6575,0.6535
cleveland,0.290397,0.2764,0.208347,0.2452,0.2384
dermatology,0.895363,0.941333,0.521995,0.95,0.94183
ecoli,0.720659,0.512,0.17585,0.529875,0.54475
glass,0.679067,0.518333,0.31381,0.521857,0.494857
heart,0.7675,0.7605,0.785833,0.818,0.7625
ionosphere,0.912468,0.8905,0.658974,0.887,0.892
pima,0.687633,0.6985,0.502775,0.7025,0.6985


In [32]:
stats.wilcoxon(no_mean['lower-new-check'], no_mean['furia'], alternative="greater")

WilcoxonResult(statistic=171.0, pvalue=3.814697265625e-06)

In [33]:
f_test = stats.friedmanchisquare(*no_mean.values)
f_test

FriedmanchisquareResult(statistic=53.48100477475607, pvalue=1.1983286540382202e-05)

In [34]:
post_hocs = sph.posthoc_conover_friedman(no_mean.values, p_adjust='Holm')
post_hocs.columns=no_mean.columns
post_hocs.index=no_mean.columns
post_hocs

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.406796,9.410312e-03,0.051265,7.314025e-04
modlem,0.406796,1.000000,5.744072e-02,0.006980,4.034528e-05
qr,0.009410,0.057441,1.000000e+00,0.000002,3.228391e-09
furia,0.051265,0.006980,1.768152e-06,1.000000,2.443989e-01
ripper,0.000731,0.000040,3.228391e-09,0.244399,1.000000e+00


In [35]:
best = no_mean.max(axis='columns')
dif = no_mean.subtract(best, axis='rows').abs()
dif.max(axis='rows').sort_values()

qr                    0.0
lower-new-check    1142.0
modlem             1164.6
furia              1512.6
ripper             1526.6
dtype: object

In [40]:
ranks = no_mean.rank(axis='columns', ascending=False)
friedman_ranks = ranks.mean()
print(friedman_ranks.sort_values().to_latex(escape=False))

\begin{tabular}{lr}
\toprule
{} &         0 \\
\midrule
furia           &  2.277778 \\
lower-new-check &  2.555556 \\
modlem          &  2.722222 \\
ripper          &  3.000000 \\
qr              &  4.444444 \\
\bottomrule
\end{tabular}


/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_74098/2662355144.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(friedman_ranks.sort_values().to_latex(escape=False))


In [152]:
ranks['lower-new-check'].value_counts()

1.0    7
4.0    6
3.0    2
2.0    2
5.0    1
Name: lower-new-check, dtype: int64

In [153]:
ranks.apply(pd.value_counts)

,lower-new-check,modlem,qr,furia,ripper
1.0,7.0,4,1.0,5.0,1
2.0,2.0,2,1.0,7.0,5
2.5,NaN,1,NaN,NaN,1
3.0,2.0,6,1.0,3.0,4
3.5,NaN,1,NaN,NaN,1
4.0,6.0,3,1.0,2.0,5
5.0,1.0,1,14.0,1.0,1


On high IR datasets (>6)

In [196]:
ir_datasets = [
    'cleveland',
    'ecoli',
    'glass',
    'spectfheart',
    'winequality-red',
    'yeast'
]

ir_no_mean = no_mean.loc[ir_datasets]
# ir_no_mean.drop('qr', axis='columns', inplace=True)

In [197]:
f_test = stats.friedmanchisquare(*ir_no_mean.values)
f_test

FriedmanchisquareResult(statistic=14.714285714285708, pvalue=0.011655524770004939)

In [198]:
post_hocs = sph.posthoc_conover_friedman(ir_no_mean.values)
post_hocs.columns=ir_no_mean.columns
post_hocs.index=ir_no_mean.columns
post_hocs

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.233052,0.004900,0.094249,0.094249
modlem,0.233052,1.000000,0.067585,0.603959,0.603959
qr,0.004900,0.067585,1.000000,0.175230,0.175230
furia,0.094249,0.603959,0.175230,1.000000,1.000000
ripper,0.094249,0.603959,0.175230,1.000000,1.000000


In [199]:
stats.wilcoxon(ir_no_mean['lower-new-check'], ir_no_mean['ripper'], alternative="greater")

WilcoxonResult(statistic=18.0, pvalue=0.078125)

In [200]:
wil_ph = sph.posthoc_nemenyi_friedman(ir_no_mean.values)
wil_ph.columns=ir_no_mean.columns
wil_ph.index=ir_no_mean.columns
wil_ph

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.680037,0.008987,0.359170,0.359170
modlem,0.680037,1.000000,0.261866,0.900000,0.900000
qr,0.008987,0.261866,1.000000,0.576422,0.576422
furia,0.359170,0.900000,0.576422,1.000000,0.900000
ripper,0.359170,0.900000,0.576422,0.900000,1.000000
